# 파지 강도 동적 계산 — v2

**담당**: 송다현, 정수진 (동적 강도 파지)

v1(`grasp.ipynb`) 재작성본. 원본은 그대로 뒀습니다.
근거: [`REVIEW.md`](REVIEW.md), 리뷰 반영: [`grasp_v2_문의점.txt`](grasp_v2_문의점.txt)

## 핵심

**로봇이 무게를 잽니다.** 유일한 실측값은 `actual_motor_torque` (`read_data_rt`) 입니다.
`get_tool_force()` / `get_workpiece_weight()` 는 **구조적으로 항상 0** 입니다
(E0509 에는 토크 센서도 F/T 센서도 없어서, `raw_joint_torque` 가 사실 `gravity_torque` 입니다).

```
① 상자 폭에 맞춰 살짝 집는다 → ② 무게를 잰다 → ③ 힘을 계산한다
                              → ④ 그 힘으로 다시 조인다 → ⑤ 들어서 확인 → ⑥ 옮긴다
```

## 실측 결과 (2026-07-14)

500ml 물병에 물을 채워가며 3점을 재서 **R² 0.9973, 오차 ±15 g** 를 얻었습니다.

```
     무게(g)     J2 토크
        20      -7.526
       270      -8.798
       520      -9.858

  질량 = -213.8 · τ(J2) - 1596.1   (g)      R² 0.9973 · RMSE 10.7 g
```

데이터: [`data/weight_calib_points.json`](data/weight_calib_points.json)

## ⚠️ 여기까지 오는 데 걸린 함정 3개

**① 짜는 반력 — 빈 그리퍼를 기준선으로 쓰면 안 됩니다**

빈 그리퍼는 stroke 에 '도달'해서 아무것도 안 누릅니다.
물체를 물면 `current` 로 계속 **누릅니다.** 그 반력이 팔에 모멘트를 만듭니다.
두 상태를 빼면 반력이 남아서 무게 신호를 덮어버립니다.

→ **기준 물체를 쓰세요.** 같은 폭·같은 힘으로 문 상태끼리 비교해야 합니다.
   (우리는 **빈 페트병 20 g** 을 기준으로 썼습니다)

**② 마찰 히스테리시스 — 팔을 건드릴 때마다 값이 달라집니다**

감속비가 커서 마찰이 **약 3 Nm**. 정지 상태의 평형 토크는 **유일하지 않습니다.**
같은 물체를 놨다 다시 물면 값이 **0.8~1.5 Nm(≈150~280 g)** 씩 튑니다.

- 가만히 두면 안정 (0.137 Nm = 양자화 한 칸)
- 하지만 **상태가 바뀌면** 새 평형점에 정착
- **dither(팔 ±1°)로 매번 새로 정착시켜 독립 표본을 만들고, 5회 평균** → 표준오차 ~10 g

**③ J2 를 쓰세요. J3 는 못 씁니다**

이 자세에서 J3 는 드리프트가 심해 R² 0.849 였습니다. **J2 는 0.9973.**
동료분의 B-4 지적대로, **어느 관절을 쓸지는 데이터로 정해야** 합니다.

## ⚠️ 캘리브레이션이 유효한 조건

```
자세      [-13.181, -23.339, 94.612, 15.686, 55.968, -15.458]
그리퍼    stroke 186 명령 · current 350
툴        rh_p12_rn (0.5 kg, cog [0,0,60])
물체 폭   30 mm
```

**하나라도 바뀌면 재캘리브레이션해야 합니다.** 특히 파지 `current` 를 바꾸면
짜는 반력이 달라져서 기존 계수가 무효가 됩니다.

## 1. 연결

```bash
./scripts/robot_up.sh
```

In [ ]:
import json
import statistics
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import rclpy

import DR_init

ROBOT_ID, ROBOT_MODEL = "dsr01", "e0509"
DR_init.__dsr__id = ROBOT_ID
DR_init.__dsr__model = ROBOT_MODEL

rclpy.init()
node = rclpy.create_node("grasp_v2", namespace=ROBOT_ID)
DR_init.__dsr__node = node          # ← DSR_ROBOT2 import 보다 먼저여야 함

from DSR_ROBOT2 import (
    posx, posj, movel, movej, wait, DR_BASE,
    set_robot_mode, get_robot_mode, get_current_posx, get_current_posj,
    set_tool, set_tcp, get_tool, get_tcp,
    read_data_rt,
    ROBOT_MODE_MANUAL, ROBOT_MODE_AUTONOMOUS,
)
from dsr_gripper import gripper_cmd

DATA = Path.cwd()          # json 저장 위치 (노트북과 같은 폴더)
print("연결 완료. 모드:", get_robot_mode(), "(1=autonomous)")

## 2. 모드 전환 — 확인하고 넘어갑니다

`set_robot_mode()` 는 **비동기**입니다. 요청만 하고 바로 다음 줄로 가면
아직 전환 전이라 명령이 조용히 실패합니다. (실제로 겪은 버그입니다)

**반드시 폴링해서 확인**한 뒤 진행합니다.

In [ ]:
def set_mode(mode, timeout=10.0):
    '''모드를 바꾸고 실제로 바뀔 때까지 기다린다. 안 바뀌면 예외.'''
    want = int(mode)
    if get_robot_mode() == want:
        return want
    set_robot_mode(want)
    t0 = time.time()
    while time.time() - t0 < timeout:
        if get_robot_mode() == want:
            name = "MANUAL" if want == 0 else "AUTONOMOUS"
            print(f"  모드 → {name} ({time.time()-t0:.1f}s)")
            return want
        time.sleep(0.3)
    raise RuntimeError(f"모드 전환 실패 (현재 {get_robot_mode()}, 원한 값 {want})")

## 3. 툴 등록 — ⚠️ 매 세션 반드시 실행하세요

**툴 등록은 `actual_motor_torque` 를 약 2.5 Nm 바꿉니다** (무게로 환산하면 **≈ 780 g**).
아이폰(0.549 Nm)의 4배가 넘습니다.

**그런데 드라이버를 재시작하면 툴 등록이 사라집니다.** (실기 확인)

```
툴 등록 → 기준선 수집 → 캘리브레이션 → (드라이버 재시작) → 툴 소실
                                       → 기준선이 2.5 Nm 어긋남
                                       → 무게가 780g 씩 틀림
```

즉 **툴 등록 상태는 기준선의 일부입니다.**
`WeightSensor` 가 기준선에 툴 이름을 함께 저장하고, 측정할 때 검사해서
**다르면 거부**합니다. 하루치 데이터가 조용히 오염되는 걸 막기 위함입니다.

> **모드 전환만으로는 토크가 안 바뀝니다** (대조 실험으로 확인 — Δ 0.000 Nm).
> 원인은 툴 등록이 맞습니다.

**manual 모드에서만** 등록됩니다.

> `cog` / TCP 는 **근사값**입니다. 그리퍼 도면으로 실측해서 채우세요.
> 무게 0.5 kg 은 ROBOTIS 공식 스펙이라 정확합니다.
> **이 값들을 바꾸면 기준선을 처음부터 다시 잡아야 합니다.**

In [ ]:
from dsr_msgs2.srv import ConfigCreateTool, ConfigCreateTcp

TOOL_NAME, TCP_NAME = "rh_p12_rn", "rh_p12_rn_tcp"
TOOL_WEIGHT = 0.5                            # kg — ROBOTIS 스펙 (확실)
TOOL_COG = [0.0, 0.0, 60.0]                  # mm — TODO(실측)
TCP_POS = [0.0, 0.0, 150.0, 0.0, 0.0, 0.0]   # mm — TODO(실측)


def _call(cli, req, what, timeout=10.0):
    '''서비스 호출 + 성공 여부 확인. 실패하면 예외.'''
    if not cli.wait_for_service(timeout_sec=5.0):
        raise RuntimeError(f"{what}: 서비스 없음")
    fut = cli.call_async(req)
    rclpy.spin_until_future_complete(node, fut, timeout_sec=timeout)
    res = fut.result()
    if res is None:
        raise RuntimeError(f"{what}: 응답 없음 (timeout)")
    if not getattr(res, "success", True):
        raise RuntimeError(f"{what}: 거부됨 (success=False). manual 모드인지 확인하세요")
    return res


def register_tool():
    ns = f"/{ROBOT_ID}/dsr_controller2"
    tool_cli = node.create_client(ConfigCreateTool, f"{ns}/tool/config_create_tool")
    tcp_cli = node.create_client(ConfigCreateTcp, f"{ns}/tcp/config_create_tcp")

    set_mode(ROBOT_MODE_MANUAL)              # ← 등록은 manual 에서만

    r = ConfigCreateTool.Request()
    r.name, r.weight, r.cog, r.inertia = TOOL_NAME, TOOL_WEIGHT, TOOL_COG, [0.0] * 6
    _call(tool_cli, r, "config_create_tool")

    r = ConfigCreateTcp.Request()
    r.name, r.pos = TCP_NAME, TCP_POS
    _call(tcp_cli, r, "config_create_tcp")

    set_tool(TOOL_NAME)
    set_tcp(TCP_NAME)

    set_mode(ROBOT_MODE_AUTONOMOUS)          # ← 그리퍼는 autonomous 에서만 동작

    tool, tcp = get_tool(), get_tcp()
    if not tool or not tcp:
        raise RuntimeError(f"툴 등록 실패 (tool={tool!r}, tcp={tcp!r})")
    print(f"✓ tool={tool}  tcp={tcp}")


register_tool()

## 4. 그리퍼 — 위치를 '지정'해서 닫습니다

| 값 | 의미 |
|---|---|
| `position` | `0` = 완전 닫힘 ~ `750` = 완전 열림 |
| `current` | **파지 힘** (뉴턴이 아니라 레지스터 raw 값) |

**v1 의 가장 큰 버그가 뉴턴과 `current` 를 섞은 것**이었습니다. 여기서는 뉴턴을 아예 안 씁니다.

기준선을 맞추려면 **손가락 위치를 알아야** 하므로, "완전히 닫아라"가 아니라
**"이 stroke 까지 닫아라"** 로 명령합니다.

In [ ]:
STROKE_CLOSED, STROKE_OPEN = 0, 750
OPEN_WIDTH_MM = 109.0      # ROBOTIS 스펙. TODO(검증): 자로 재서 확인하세요

CURRENT_MIN = 50           # 이보다 낮으면 아예 못 잡음
CURRENT_MAX = 500          # TODO(실험): 상자가 찌그러지기 시작하는 값
CURRENT_PROBE = 100        # 무게 잴 때 '살짝' 잡는 힘. TODO(실험): 물체마다 적정값 확인

SQUEEZE_MM = 3.0           # 상자 폭보다 이만큼 더 조여서 확실히 문다


def stroke_from_width_mm(w_mm):
    '''개구 폭(mm) → stroke(0~750).'''
    s = round(w_mm / OPEN_WIDTH_MM * STROKE_OPEN)
    return int(max(STROKE_CLOSED, min(STROKE_OPEN, s)))


def width_mm_from_stroke(s):
    return max(0, min(STROKE_OPEN, s)) / STROKE_OPEN * OPEN_WIDTH_MM


def clamp_current(c, context=""):
    '''안전 범위로 자르되, MAX 초과는 '위험 신호'이므로 크게 경고한다.

    MIN 미만  : 계산값이 너무 약함 → 올림 (정보)
    MAX 초과  : 이 물체는 지금 설정으로 못 버틸 수 있음 → 경고 (위험)
    '''
    c = round(c)
    if c > CURRENT_MAX:
        print(f"  ⚠️ [{context}] 필요 힘 {c} 이 CURRENT_MAX({CURRENT_MAX})를 넘습니다.")
        print(f"     → 이 물체는 지금 설정으로 놓칠 수 있습니다. 상자가 찌그러지지 않는")
        print(f"        선에서 CURRENT_MAX 를 올리거나, 이 물체는 포기해야 합니다.")
        return CURRENT_MAX, "over_max"
    if c < CURRENT_MIN:
        print(f"  · [{context}] 계산값 {c} → 최소값 {CURRENT_MIN} 으로 올림")
        return CURRENT_MIN, "under_min"
    return int(c), "ok"


def grip_at(stroke, current, context="grip"):
    '''지정한 stroke 까지, 지정한 힘으로 닫는다.'''
    cur, _ = clamp_current(current, context)
    gripper_cmd(int(stroke), current=cur)
    wait(1)
    return cur


def release():
    gripper_cmd(STROKE_OPEN, current=CURRENT_MIN)
    wait(1)


print(f"stroke 0~{STROKE_OPEN} ↔ 0~{OPEN_WIDTH_MM}mm   current {CURRENT_MIN}~{CURRENT_MAX}")

## 5. 무게 측정 — 실측으로 확정된 방식

```
질량 = a · τ(J2) + b        ← 절대 토크. 기준선을 따로 빼지 않습니다
```

기준선을 빼지 않는 이유: **조건이 전부 같으면 기준선이 절편 `b` 에 흡수**됩니다.
빈 그리퍼 기준선은 짜는 반력이 없어서 오히려 오염원이 됩니다 (위 함정 ①).

### 절차

```
① 기준 물체(빈 페트병 등)를 물고 dither+평균  →  캘리브레이션 점
② 무게를 바꿔가며 3~4점                       →  회귀 (R² 확인)
③ 이후 아무 물체나 물고 measure()             →  질량
```

**모든 점을 같은 폭·같은 stroke·같은 current·같은 자세**에서 재야 합니다.

### dither 가 필요한 이유

마찰 때문에 정지 평형이 유일하지 않습니다.
**가만히 두면 값이 안 변하고**(양자화 한 칸), **건드리면 새 평형점으로 튑니다**(0.8~1.5 Nm).

dither 로 **매번 새로 정착**시켜 독립 표본을 만들고 **평균**을 내면 오차가 √N 으로 줄어듭니다.
5회 평균 → 표준오차 약 **10 g**.

In [ ]:
TORQUE_SAMPLES = 25          # 한 번 잴 때 샘플 수
CYCLES = 5                   # dither 반복 (평균낼 독립 표본 수)
DITHER_JOINT = 1             # J2 (0-based) — dither 로 흔들 관절
DITHER_DEG = 1.0
DITHER_VEL = 15              # deg/s — 물체가 매달려 있으므로 천천히

WEIGHT_JOINT = 1             # J2. 실측 결과 J3 는 드리프트로 못 씀 (R² 0.849)
CALIB_PATH = DATA / "weight_calib.json"


def motor_torque(n=TORQUE_SAMPLES):
    """actual_motor_torque 중앙값 (전체 관절)."""
    rows = []
    for _ in range(n):
        d = read_data_rt()
        if d is not None:
            rows.append(list(d.actual_motor_torque))
        time.sleep(0.04)
    if not rows:
        raise RuntimeError("read_data_rt 가 데이터를 주지 않습니다")
    return [statistics.median(c) for c in zip(*rows)]


class WeightSensor:
    """모터 토크로 무게를 잰다. (2026-07-14 실기 검증: R² 0.9973, ±15 g)

    질량 = a·τ(J2) + b   — 절대 토크. 기준선을 빼지 않습니다.
    조건(자세·stroke·current·툴)이 전부 같아야 성립합니다.
    """

    def __init__(self):
        self.points = []          # [{mass_kg, tau, sem}]
        self.a = self.b = None
        self.cond = None
        self._load()

    # ── 조건 검증 ───────────────────────────────────────
    def _now_cond(self, stroke, current):
        return {"tool": get_tool(), "stroke": int(stroke), "current": int(current),
                "pose": [round(v, 3) for v in get_current_posj()]}

    def _check_cond(self, stroke, current):
        """조건이 캘리브레이션 때와 같은지 검사. 다르면 거부."""
        if self.cond is None:
            return
        now = self._now_cond(stroke, current)
        for k in ("tool", "stroke", "current"):
            if now[k] != self.cond[k]:
                raise RuntimeError(
                    f"{k} 가 다릅니다 (캘리브 {self.cond[k]!r} → 지금 {now[k]!r}).\n"
                    "  조건이 바뀌면 짜는 반력·중력이 달라져 계수가 무효입니다.\n"
                    "  → 재캘리브레이션하세요.")
        dq = max(abs(x - y) for x, y in zip(now["pose"], self.cond["pose"]))
        if dq > 0.5:
            raise RuntimeError(f"자세가 {dq:.2f}° 달라졌습니다. 재캘리브레이션하세요.")

    # ── 저장/로드 ───────────────────────────────────────
    def _load(self):
        if not CALIB_PATH.exists():
            return
        d = json.loads(CALIB_PATH.read_text())
        self.points, self.a, self.b, self.cond = d["points"], d["a"], d["b"], d["cond"]
        if self.a is not None:
            print(f"캘리브레이션 로드: 질량 = {self.a*1000:.1f}·τ {self.b*1000:+.1f} g "
                  f"({len(self.points)}점)")

    def _save(self):
        CALIB_PATH.write_text(json.dumps(
            {"points": self.points, "a": self.a, "b": self.b, "cond": self.cond},
            indent=2, ensure_ascii=False))

    # ── 측정 (dither + 평균) ────────────────────────────
    def read(self, cycles=CYCLES):
        """dither 로 매번 새로 정착시키며 여러 번 재고 평균 → (평균, 표준오차)

        ⚠️ 팔이 움직입니다 (J2 를 DITHER_DEG 만큼 갔다가 복귀).
        """
        q0 = [round(v, 3) for v in get_current_posj()]
        vals = []
        for i in range(cycles):
            away = list(q0)
            away[DITHER_JOINT] += DITHER_DEG
            movej(posj(away), vel=DITHER_VEL, acc=DITHER_VEL)
            wait(0.4)
            movej(posj(q0), vel=DITHER_VEL, acc=DITHER_VEL)   # 항상 같은 방향에서 복귀
            wait(1.2)
            t = motor_torque()
            vals.append(t[WEIGHT_JOINT])
            print(f"    {i+1}/{cycles}  tau(J2) = {t[WEIGHT_JOINT]:+8.3f}")

        m = statistics.mean(vals)
        sd = statistics.pstdev(vals) if len(vals) > 1 else 0.0
        return m, sd / max(len(vals) ** 0.5, 1)

    # ── 캘리브레이션 ────────────────────────────────────
    def add_point(self, known_mass_kg, stroke, current):
        """무게를 아는 물체를 문 상태에서 실행. 3~4점 모으세요."""
        if self.cond is None:
            self.cond = self._now_cond(stroke, current)
            print(f"  조건 기록: stroke {self.cond['stroke']}, current {self.cond['current']}, "
                  f"tool {self.cond['tool']!r}")
        else:
            self._check_cond(stroke, current)

        tau, sem = self.read()
        self.points.append({"mass_kg": float(known_mass_kg), "tau": tau, "sem": sem,
                            "t": datetime.now().isoformat(timespec="seconds")})
        self._save()
        print(f"\n  [{len(self.points)}] {known_mass_kg*1000:.0f} g → tau {tau:+.3f} Nm "
              f"(표준오차 {sem:.3f})")

    def fit(self):
        """질량 = a·τ + b 회귀 + R² + LOO-CV."""
        n = len(self.points)
        if n < 3:
            print(f"점이 {n}개 — 최소 3개 필요합니다 (선형성 확인용)")
            return None

        x = np.array([p["tau"] for p in self.points])
        y = np.array([p["mass_kg"] for p in self.points])
        A = np.vstack([x, np.ones_like(x)]).T
        (self.a, self.b), *_ = np.linalg.lstsq(A, y, rcond=None)

        pred = self.a * x + self.b
        resid = pred - y
        rmse = float(np.sqrt(np.mean(resid ** 2)))
        r2 = 1 - np.sum(resid ** 2) / max(np.sum((y - y.mean()) ** 2), 1e-12)

        print(f"\n  질량 = {self.a*1000:.1f}·tau {self.b*1000:+.1f}   (g)")
        print(f"  {n}점 · RMSE {rmse*1000:.1f} g · R2 {r2:.4f}\n")
        for p, pr in zip(self.points, pred):
            print(f"    실제 {p['mass_kg']*1000:>5.0f} g → 예측 {pr*1000:>6.0f} g   "
                  f"오차 {(pr-p['mass_kg'])*1000:+6.0f} g")

        if n >= 4:
            errs = []
            for i in range(n):
                k = [j for j in range(n) if j != i]
                Ai = np.vstack([x[k], np.ones(len(k))]).T
                (ai, bi), *_ = np.linalg.lstsq(Ai, y[k], rcond=None)
                errs.append(abs(ai * x[i] + bi - y[i]))
            print(f"\n  LOO-CV 평균오차 {np.mean(errs)*1000:.1f} g")

        if r2 < 0.95:
            print("\n  ⚠️ R2 가 낮습니다. 물체가 미끄러졌거나 조건이 흔들렸을 수 있습니다.")
            print("     젖은 물체는 current 350 으로도 미끄러집니다 — 실측에서 겪었습니다.")

        self._save()
        return self.a, self.b

    # ── 사용 ────────────────────────────────────────────
    def measure(self, stroke, current):
        """지금 물고 있는 물체의 질량(kg)과 신뢰 여부 → (mass_kg, ok)"""
        if self.a is None:
            raise RuntimeError("캘리브레이션이 없습니다. add_point() x3 → fit() 하세요")
        self._check_cond(stroke, current)

        tau, sem = self.read()
        mass = self.a * tau + self.b
        err_g = abs(self.a * sem) * 1000
        ok = mass > 0 and err_g < 50
        print(f"\n  tau {tau:+.3f} Nm  →  {mass*1000:.0f} g  (±{err_g:.0f} g)")
        if not ok:
            print("  ⚠️ 신뢰할 수 없습니다. 물체가 매달려 있고 안 미끄러지는지 확인하세요.")
        return max(0.0, mass), ok


scale = WeightSensor()

## 6. 픽 자세 티칭 — 파일로 저장됩니다

> ⚠️ **이 자세를 세션 내내 고정하세요.** 무게 측정이 자세에 의존합니다.
> 자세가 바뀌면 기준선 테이블과 캘리브레이션을 **처음부터 다시** 해야 합니다.

In [ ]:
PICK_PATH = DATA / "pick_pose.json"
PICK_POS = json.loads(PICK_PATH.read_text())["pick"] if PICK_PATH.exists() else None
if PICK_POS:
    print("저장된 픽 자세 로드:", PICK_POS)
else:
    print("저장된 픽 자세 없음 — 아래 두 셀로 티칭하세요")

In [ ]:
set_mode(ROBOT_MODE_MANUAL)
print("직접교시로 팔을 Pick 위치로 옮긴 뒤 다음 셀을 실행하세요.")

In [ ]:
_x, _ = get_current_posx()
PICK_POS = [round(v, 3) for v in _x]
PICK_PATH.write_text(json.dumps({"pick": PICK_POS,
                                 "t": datetime.now().isoformat(timespec="seconds")}, indent=2))
print("PICK_POS =", PICK_POS, "→ 저장됨 (커널 재시작해도 유지)")

set_mode(ROBOT_MODE_AUTONOMOUS)

## 7. 무게 캘리브레이션 — **오늘의 첫 관문**

무게를 **저울로 잰** 물체 **3~4개**로 회귀합니다.

**가장 좋은 방법: 500 ml 물병에 물을 채워가며 재세요.**
폭이 안 바뀌므로 stroke·기준·자세가 전부 고정되고, **오직 무게만** 변합니다.

```
빈 병    20 g
물 절반  270 g
물 가득  520 g
```

> ⚠️ **물기를 반드시 닦으세요.** 젖은 병은 `current 350` 으로도 미끄러집니다.
> 미끄러지면 값이 조용히 오염됩니다 (실측에서 두 번 겪었습니다).

In [ ]:
# 물체 하나마다 이 셀을 실행하세요
KNOWN_MASS_KG = 0.020      # 저울로 잰 값
WIDTH_MM = 30.0            # 자로 잰 폭

stroke = stroke_from_width_mm(WIDTH_MM - SQUEEZE_MM)
release()
input(f"물체를 그리퍼 사이에 놓고 Enter (stroke {stroke} · current {CURRENT_PROBE} 로 뭅니다)")
grip_at(stroke, CURRENT_PROBE, context="calib")
time.sleep(1)

scale.add_point(KNOWN_MASS_KG, stroke, CURRENT_PROBE)   # ⚠️ 팔이 dither 로 움직입니다

In [ ]:
# 3~4점 모았으면 실행
scale.fit()

## 9. 파지 강도 계산

물리식이 **형태**를 줍니다: `F ≥ k·M·g / (2·μ)` → **필요한 힘은 무게에 비례**.

`current` 는 힘에 대략 비례하므로:

```
current = A · 질량(kg) + B
```

`A`, `B` 는 **실험으로** 구합니다. **뉴턴을 거치지 않습니다** — N ↔ current 변환은 알 수 없고,
알 필요도 없습니다.

In [ ]:
SAFETY = 1.2          # 안전 계수. 최소값에 딱 맞추면 실제로는 미끄러집니다
FIXED_CURRENT = 250   # 데이터가 없거나 무게를 못 믿을 때 쓰는 보수적 값


class GraspPlanner:
    def __init__(self):
        self.a = self.b = None

    def current_for(self, mass_kg=None, trustworthy=True):
        '''파지 current 를 정한다. (current, mode, status) 반환.

        무게를 못 믿거나(trustworthy=False) 계수가 없으면 fixed 로 자동 폴백.
        '''
        if self.a is None or mass_kg is None or not trustworthy:
            why = ("계수 없음" if self.a is None
                   else "무게 없음" if mass_kg is None else "무게 신뢰 불가")
            print(f"  → fixed 모드로 폴백 ({why}): current {FIXED_CURRENT}")
            c, st = clamp_current(FIXED_CURRENT, "fixed")
            return c, "fixed", st

        raw = (self.a * mass_kg + self.b) * SAFETY
        c, st = clamp_current(raw, "measured")
        return c, "measured", st

    def fit(self, trials):
        '''accept 된 (질량, 최소성공 current) 로 A, B 회귀 + LOO-CV.'''
        ok = [t for t in trials if t.get("accepted")]
        if len(ok) < 2:
            print(f"accept 된 데이터가 {len(ok)}개 — 2개 이상 필요합니다")
            return None
        m = np.array([t["mass_kg"] for t in ok])
        c = np.array([float(t["current"]) for t in ok])
        A = np.vstack([m, np.ones_like(m)]).T
        (self.a, self.b), *_ = np.linalg.lstsq(A, c, rcond=None)

        pred = self.a * m + self.b
        resid = pred - c
        rmse = float(np.sqrt(np.mean(resid ** 2)))
        r2 = 1 - np.sum(resid ** 2) / max(np.sum((c - c.mean()) ** 2), 1e-12)
        print(f"\n  current = {self.a:.0f}·M {self.b:+.0f}   (데이터 {len(ok)}점)")
        print(f"  RMSE {rmse:.1f} · R² {r2:.4f}")

        if len(ok) >= 4:
            errs = []
            for i in range(len(ok)):
                k = [j for j in range(len(ok)) if j != i]
                Ai = np.vstack([m[k], np.ones(len(k))]).T
                (ai, bi), *_ = np.linalg.lstsq(Ai, c[k], rcond=None)
                errs.append(abs(ai * m[i] + bi - c[i]))
            print(f"  LOO-CV 평균오차 {np.mean(errs):.1f} current")

        if r2 < 0.9:
            print("  ⚠️ R² 가 낮습니다. 판정이 애매했던 시도를 다시 보세요.")
        return self.a, self.b


planner = GraspPlanner()

## 10. 파지 시퀀스 — 들어올린 뒤 확인까지

```
① 상자 폭에 맞춰 살짝 집는다 → ② 무게를 잰다 → ③ 힘을 계산한다
                              → ④ 그 힘으로 다시 조인다 → ⑤ 들어올린다
                              → ⑥ 다시 재서 '아직 들고 있는지' 확인   ← 새로 추가
```

⑥ 이 없으면 **이동 중에 떨어뜨려도 모릅니다.** 정지 상태보다 이동 중(가속·감속)이
더 큰 힘을 요구하므로 오히려 더 위험한 구간입니다.

In [ ]:
LIFT_MM = 100.0
SPEED_L = 50


def move_to(pose, vel=SPEED_L):
    movel(posx(list(pose)), vel=vel, acc=vel, ref=DR_BASE)
    wait(1)


def pick_with_weight_feedback(width_mm, pick_pos=None, place_pos=None):
    '''살짝 집기 → 무게 재기 → 힘 계산 → 다시 조이기 → 들어서 확인 → 옮기기.'''
    pick_pos = pick_pos or PICK_POS
    stroke = stroke_from_width_mm(width_mm - SQUEEZE_MM)

    # ① 접근 + 살짝 집기
    release()
    move_to(pick_pos)
    grip_at(stroke, CURRENT_PROBE, context="probe")
    time.sleep(1)

    # ② 무게 측정 (픽 자세 그대로)
    mass, ok = scale.measure(stroke, CURRENT_PROBE)

    # ③ 파지력 계산 — 무게를 못 믿으면 자동으로 fixed 폴백
    cur, mode, status = planner.current_for(mass if ok else None, trustworthy=ok)
    print(f"  질량 {mass*1000:.0f} g ({'신뢰' if ok else '불신'}) → current {cur} [{mode}]")

    # ④ 그 힘으로 다시 조인다
    grip_at(stroke, cur, context="grip")
    time.sleep(1)

    # ⑤ 들어올린다
    lifted = list(pick_pos)
    lifted[2] += LIFT_MM
    move_to(lifted)

    # ⑥ 아직 들고 있나? — 놓쳤으면 여기서 잡힙니다
    held, held_ok = scale.measure(stroke, CURRENT_PROBE)
    if not held_ok or held < mass * 0.5:
        print(f"  ❌ 물체를 놓쳤습니다! (집을 때 {mass*1000:.0f}g → 지금 {held*1000:.0f}g)")
        if status == "over_max":
            print("     CURRENT_MAX 에 걸렸습니다. 이 물체는 지금 설정으로 못 듭니다.")
        return {"mass_kg": mass, "current": cur, "mode": mode, "held": False}

    print(f"  ✓ 들고 있습니다 ({held*1000:.0f}g)")

    if place_pos is not None:
        move_to(place_pos)
        release()
        up = list(place_pos)
        up[2] += LIFT_MM
        move_to(up)

    return {"mass_kg": mass, "current": cur, "mode": mode, "held": True}

## 11. 실험 루프 — `A`, `B` 를 구하는 데이터를 만든다

**오늘의 본 게임입니다.** 계수가 없으면 `measured` 모드가 안 켜집니다.

상자 하나마다:

```python
try_current(150, width_mm=70)    # 낮은 값부터
reject(0)                         # 미끄러졌으면
try_current(200, width_mm=70)
accept(1)                         # 버텼으면
```

**"버틴 것 중 가장 낮은 `current`" 를 accept 하세요.** 아무 성공값이나 넣으면 회귀가 과대평가됩니다.

무게를 바꿔가며 **최소 3종** 반복하세요.

> 상자가 **찌그러지기 시작하는 `current`** 도 `note` 에 남기세요 → `CURRENT_MAX` 를 그 아래로.

In [ ]:
TRIALS_PATH = DATA / "grasp_trials_v2.json"
trials = json.loads(TRIALS_PATH.read_text()) if TRIALS_PATH.exists() else []


def _save():
    TRIALS_PATH.write_text(json.dumps(trials, indent=2, ensure_ascii=False))


def try_current(current, width_mm, pick_pos=None, note=""):
    '''그 힘으로 집어 들어본다. 무게·폭·시각이 함께 기록된다.'''
    pick_pos = pick_pos or PICK_POS
    stroke = stroke_from_width_mm(width_mm - SQUEEZE_MM)

    release()
    move_to(pick_pos)

    grip_at(stroke, CURRENT_PROBE, context="probe")   # 먼저 살짝 집어 무게를 잰다
    time.sleep(1)
    mass, ok = scale.measure(stroke, CURRENT_PROBE)

    cur = grip_at(stroke, current, context="trial")    # 시험할 힘으로 다시 조인다
    time.sleep(1)

    lifted = list(pick_pos)
    lifted[2] += LIFT_MM
    move_to(lifted)

    t = {"index": len(trials), "mass_kg": mass, "mass_ok": ok,
         "width_mm": width_mm, "stroke": stroke, "current": int(cur),
         "accepted": None, "note": note,
         "t": datetime.now().isoformat(timespec="seconds")}
    trials.append(t)
    _save()

    print(f"\n[{t['index']}] 질량 {mass*1000:.0f} g · 폭 {width_mm}mm · current {cur}")
    print("  → 미끄러졌나요? accept(i) / reject(i) 로 판정하세요.")
    return t["index"]


def review():
    print(f"{'idx':>3} | {'질량(g)':>8} | {'폭':>5} | {'current':>7} | {'판정':>8} | note")
    for t in trials:
        v = {True: "버팀", False: "미끄러짐", None: "-"}[t["accepted"]]
        w = "" if t.get("mass_ok", True) else " ⚠무게불신"
        print(f"{t['index']:>3} | {t['mass_kg']*1000:>8.0f} | {t['width_mm']:>5.0f} | "
              f"{t['current']:>7} | {v:>8} | {t['note']}{w}")


def accept(i, note=""):
    if not trials[i].get("mass_ok", True):
        print(f"  ⚠️ [{i}] 은 무게를 신뢰할 수 없는 시도입니다. 회귀가 오염됩니다.")
    trials[i]["accepted"] = True
    if note:
        trials[i]["note"] = note
    _save()
    print(f"✅ [{i}] 버팀 — 회귀에 사용됩니다")


def reject(i, note=""):
    trials[i]["accepted"] = False
    if note:
        trials[i]["note"] = note
    _save()
    print(f"❌ [{i}] 미끄러짐 — 기록만 남김")


print(f"이전 시도 {len(trials)}건 로드됨")

In [ ]:
# 다 모았으면 회귀
review()
planner.fit(trials)

## 12. 실제 파지

```python
pick_with_weight_feedback(width_mm=70)                  # 무게 기반 동적 파지
pick_with_weight_feedback(width_mm=70, place_pos=...)   # 옮기기까지
```

## 백로그 (오늘 안 함)

- **다중 관절 결합** — 지금은 J3만 씁니다. `full_dtau` 를 이미 기록하고 있으니,
  데이터가 쌓이면 J2+J3 결합으로 SNR 을 올릴 수 있는지 **데이터로 판단**하면 됩니다.
- **점진적 그립 + 실시간 슬립 감지** — 제약이 있습니다:
  - 그리퍼 **현재 위치를 읽을 수 없습니다** (`GripperCmd.srv` = 제어 전용, 피드백 없음).
    쓰려면 `gripper_service` 에 Modbus 읽기(FC03, Present Position=611)를 추가해야 합니다.
  - 슬립 신호로 **모터 토크**는 쓸 수 있지만, `motor_torque()` 가 **약 2초** 걸립니다.
    이동 중 실시간 감지엔 느립니다. 샘플 수를 줄이면 노이즈가 커지고요 — **트레이드오프를
    실측으로 정해야** 합니다.

In [ ]:
# pick_with_weight_feedback(width_mm=70)

## 13. 이동 안전성 테스트 (v3 추가분) — v2 는 한 글자도 안 건드립니다

**여기부터가 새로 추가된 셀입니다.** 위 1~12절은 [`grasp_v2.ipynb`](grasp_v2.ipynb) 와
**완전히 동일**합니다 — 실측으로 검증된 무게 측정·파지력 계산은 그대로 믿고 그 위에
이어붙입니다.

목적은 **"이동하는 동안 떨어뜨리거나 미끄러지지 않는지"** 검증하는 것입니다.
`TransportTest` 로 기능을 단계별로 켜서 테스트합니다. `weight_calib.json` 이 없어도
(=`scale.measure()` 를 안 쓰므로) 바로 돌릴 수 있습니다.

```python
TransportTest(move='1')                     # 1단계: 근거리 단순 이동만
TransportTest(move='1', vel='measure')      # 1단계 + 이동 전후 토크 기록
TransportTest(move='2')                     # 2단계: 체크포인트 이동 (놓침 감지 켜짐)
TransportTest(move='3', place_pos=[...])    # 3단계: 실좌표(OCR) 이동
```

`move` 는 **낮은 단계부터 순서대로** 통과시키세요. 각 단계 모두 리프트 후 "잡혔는지"
coarse 재시도(`retry='grip'`, 기본 켜짐)와 `release_confirmed()`(release 재전송)를
같이 검증합니다.

### ⚠️ 아래 상수들은 전부 미검증 추정치입니다 — 표를 채우면서 검증하세요

| 상수 | 지금 값 | 무엇을 실측해야 하나 | 어느 단계에서 |
|---|---|---|---|
| `DROP_THRESHOLD_NM` | 1.8 Nm | 정상 이동에서 오탐 안 나는 최소값 / 실제로 놓쳤을 때 못 잡지 않는 최대값 (마찰 밴드 ~1.5 Nm 보다 커야 함, `FINDINGS_85mm.md` §4) | `move='2'`/`'3'` |
| `QUICK_CHECK_SETTLE_S` | 0.4 s | 구간 도착 직후 동적 성분(가속·감속 잔향)이 빠지는 데 걸리는 시간 | `move='2'`/`'3'` |
| `TRANSIT_CHECKPOINTS` | 2 | 늘리면 탐지는 빨라지지만 왕복 오버헤드 증가 — 트레이드오프 | `move='2'`/`'3'` |
| `RELEASE_RETRIES` / `RELEASE_RETRY_WAIT_S` | 2회 / 0.5 s | 재전송이 실제로 "명령 유실"을 고치는지, 재전송 성공률 | 항상 (`release_confirmed` 매번 실행) |
| `CURRENT_STEP` | 50 | 리프트 재시도 시 증분이 적절한지, `CURRENT_MAX` 와의 관계 | `retry='grip'` (기본값) |
| 수평 이동 속도 (`SPEED_L`=50 고정) | — | place 방향(수평) 이동에서도 안전한지 — 지금까지는 수직 리프트에서만 암묵적으로 검증됨 (v2 11절 `try_current`) | `move='2'`/`'3'`, `vel='measure'` 로 기록 |

> **속도/가속도를 별도로 스윕할 필요는 없다고 판단했습니다.** 무게 기반 자동 속도
> 조절 같은 최적화는 지금보다 더 빠르게 갈 계획이 없는 한 급하지 않습니다. 정말
> 확인이 필요한 건 "이미 쓰는 속도(`SPEED_L`)가 **수평**
> 이동에서도 안전한가" 하나뿐이라, 별도 실험 대신 `move='2'`/`'3'` 를 돌리면서
> `vel='measure'` 로 데이터를 쌓는 걸로 충분합니다.

`vel='measure'` 로 실행하면 `transport_test_log.json` 에 폭·속도·이동 전후 토크가
쌓입니다. 몇 번 돌린 뒤 실제로 미끄러졌는지(영상·육안 확인)를 `mark_slip()` 으로
표시해두면, 나중에 `DROP_THRESHOLD_NM` 이나 속도 상한을 그 데이터로 정할 수 있습니다.

### ⚠️ OCR팀 코드에서 확인한 것 — 좌표 인터페이스

`doosan_ws~/yh/` (실제 repo, `hand_eye_calib.py`/`click_and_move.py`) 확인 결과:

```
위치: 로봇 base 기준 XYZ (mm)
방향: 고정 하향 GRIPPER_DOWN_ORI_DEG = [0.0, 180.0, 0.0]
작업공간: WORKSPACE_MIN_MM ~ WORKSPACE_MAX_MM (아래 상수로 반영함, OCR팀 실측값 그대로)
한 번 이동 제한: MAX_STEP_MM = 400
```

**작업공간 범위·한 번 이동 제한은 `check_workspace()`로 코드에 반영했습니다** (`move='2'`/`'3'`
에서 자동 검사). 우리가 추측한 값이 아니라 OCR팀이 실제로 쓰는 값을 그대로 가져온
거라 지금 넣어도 됩니다.

**방향(`[0, 180, 0]` 고정 하향)은 코드로 안 넣었습니다.** 지금 `PICK_POS`는 기울어진
자세로 무게 캘리브레이션이 그 자세에 고정돼 있어서, 하향 자세로 바꾸면 캘리브레이션
자체가 무효가 될 수 있습니다. **"픽업 자세를 OCR팀 방식(하향 고정)에 맞출지, 지금
방식(기울어진 고정 자세)을 유지할지"는 팀 논의가 필요합니다** — 후자라면 물체 위치가
매번 달라질 때 무게 측정이 어떻게 되는지도 같이 정해야 합니다 (앞서 나온 "픽업 위치
고정 여부" 질문과 같은 맥락).

### 🐛 최종 점검에서 발견한 것 (2026-07-15, 로봇 테스트 전)

- **버그(수정함)**: `_grip_and_lift()` 의 `empty_tau` 가 `PICK_POS` 도착 *전*에 측정되고
  있었습니다. J2 토크는 자세에 따라 달라지는데 기준선 자세가 안 맞을 수 있어서,
  `move_to(PICK_POS)` 뒤로 옮겼습니다.
- **⚠️ 한계(미해결)**: `move_to_checked()`/`release_confirmed()` 는 `held_tau`(PICK_POS
  근처에서 잰 값)를 **목적지에서 잰 값과 그대로 비교**합니다. J2 토크는 "물체 유무"뿐
  아니라 **팔이 얼마나 뻗어있는지(자세)** 에도 좌우되므로, 짧은 이동(`move='1'`)에선
  무시할 만해도 **긴 이동(`move='2'`/`'3'`, 특히 수원·서울처럼 수백 mm 밖)에서는
  자세 때문에 생기는 토크 변화가 `DROP_THRESHOLD_NM`보다 커질 수 있습니다.**
  → 아무것도 안 놓쳤는데 `DroppedError`/release 재전송이 계속 뜨는 오탐 가능성.
  **완화한 것**: `move_to_checked()` 가 이제 원점 대신 **직전 구간 대비 급변**만
  봅니다(자세 drift는 완만, 진짜 놓침은 급격 — 완전 해결은 아니지만 도움 됨).
  **진단용으로 추가한 것**: `gravity_corrected_tau()` — 로봇이 계산하는 중력 모델값
  (`raw_joint_torque`)을 빼서 자세 성분을 줄여보는 함수. `quick_torque_check()` 와
  나란히 찍어서 어느 쪽이 더 안정적인지 실기에서 비교해보세요. 완전한 해결책은
  아닙니다 — `TOOL_COG` 가 근사값이고, 마찰 히스테리시스(~3 Nm, 이 프로젝트
  핵심 노이즈원)는 중력 모델이 애초에 못 잡습니다.

### 아직 코드로 안 만든 것

- **이동 중 놓쳤을 때 어디로 갈지** — 지금은 `DroppedError` 가 나면 **그 자리에서
  멈춥니다** (다른 위치로 옮기는 로직 없음). 안전 위치가 필요한지, 필요하면 어디인지는
  팀 논의가 필요해서 비워뒀습니다.
  > 아이디어(미구현, 관측 후 추가 검토): `DroppedError` 잡은 직후 `quick_torque_check()`
  > 를 한 번 더 재서 grip 전 `empty_tau`(빈 그리퍼 기준선)와 비교 — 거의 같으면
  > "완전히 놓친 것"으로 보고 `PICK_POS` 로 복귀, 다르면 "뭔가 불안정하게 물려있는
  > 것"으로 보고 그 자리에서 정지. 지금 안 넣은 이유: `DROP_THRESHOLD_NM=1.8` 이
  > 마찰 밴드보다 넉넉해서 `SPEED_L=50` 처럼 느린 이동에선 `DroppedError` 자체가
  > 실제로 거의 안 터질 수 있음 — `move='1'`/`'2'` 로 먼저 관측하고, 실제로 발생하면
  > 그때 넣기로 함.
- **place 후 넘어짐 확인** — `release_confirmed()` 는 "그리퍼에서 빠졌는지"만 보고,
  물체가 바닥에 안정적으로 섰는지는 비전 쪽 확인이 필요합니다 (데이터/분류 파트 연결점).

In [ ]:
# ── 이동 안전성 유틸 — v2 함수(move_to, grip_at, release, motor_torque 등)를 그대로 재사용 ──

DROP_THRESHOLD_NM = 1.8      # TODO(실험): 마찰 밴드(~1.5 Nm, FINDINGS_85mm.md §4)보다 크게
QUICK_CHECK_SAMPLES = 5      # 빠른 확인용 샘플 수 (dither 없이, ~0.3초)
QUICK_CHECK_SETTLE_S = 0.4   # 구간 도착 후 읽기 전 대기. TODO(실험)
TRANSIT_CHECKPOINTS = 2      # 이동 경로를 몇 토막으로 나눠 확인할지. TODO(실험)

RELEASE_RETRIES = 2          # release 명령이 안 먹혔을 때 재전송할 횟수
RELEASE_RETRY_WAIT_S = 0.5   # 재전송 전 대기

# 작업공간 안전 범위 — 우리가 추측한 값이 아니라 OCR팀 yh/click_and_move.py 에서
# 실제로 쓰고 있는 값을 그대로 가져옴 (doosan_ws~/yh/click_and_move.py)
WORKSPACE_MIN_MM = [-700.0, -700.0, 50.0]
WORKSPACE_MAX_MM = [700.0, 700.0, 900.0]
MAX_STEP_MM = 400.0          # 한 번에 허용할 최대 이동 거리 (OCR팀과 동일)

CURRENT_STEP = 50            # 리프트 재시도마다 올릴 current. TODO(실험)

TEST_CURRENT = FIXED_CURRENT     # 검증용 고정 current. scale 없이 이 값으로만 문다
SHORT_MOVE_OFFSET_MM = 100.0     # move='1' 기본 이동 거리. 예시값
WAYPOINT_OFFSET_MM = 300.0       # move='2' 기본 이동 거리 (place_pos 안 주면 사용)

TRANSPORT_LOG_PATH = DATA / "transport_test_log.json"

# OCR팀 놓기 구역 (doosan_ws~/yh/ocr_click_and_move.py PLACE_ZONES) — X/Y만 가져옴.
# ⚠️ "경기"는 코드에 없음 (README는 3곳이라 했지만 실제 구현은 수원/서울 2곳뿐 — 팀 확인 필요)
# ⚠️ 수원 Y좌표: 코드 주석은 "(350,500)", 실제 PLACE_ZONES 값은 cy=-500.0 — 불일치, 팀 확인 필요
# Z는 OCR팀 쪽에서 픽업 시점 값(z_saved)을 그대로 쓰는 동적 값이라 못 가져옴 —
# 여기서는 우리 PICK_POS 기준 Z를 그대로 씁니다.
PLACE_ZONE_XY = {
    "수원": (350.0, -500.0),   # PLACE_ZONES 코드값 기준. 주석과 다름 — 확인 필요
    "서울": (370.0, 0.0),
}

# OCR팀 고정 하향 자세 (yh/doosan_client.py GRIPPER_DOWN_ORI_DEG 그대로) — 우리가
# 추측한 값 아님. 자세 선택은 아직 팀 논의 전이라, 안전한 쪽(PICK_POS)을 기본값으로
# 하고 이건 명시적으로 ori='down' 을 넘겨야만 쓰이게 해뒀습니다.
GRIPPER_DOWN_ORI_DEG = [0.0, 180.0, 0.0]


def resolve_place_pos(keyword, z=None, ori=None):
    '''OCR팀 놓기 구역 키워드('수원'/'서울') → place_pos 6-vector 로 변환.

    ori
        None(기본)   PICK_POS 자세 그대로 — 우리 무게 캘리브레이션과 호환 (안전)
        'down'       OCR팀 고정 하향 자세(GRIPPER_DOWN_ORI_DEG) — 무게 캘리브레이션
                     무효화될 수 있음. 팀에서 자세를 하향으로 통일하기로 정하면 사용
        [r,p,y]      직접 지정
    '''
    if keyword not in PLACE_ZONE_XY:
        raise ValueError(f"'{keyword}' 는 등록된 구역이 아닙니다 (사용 가능: {list(PLACE_ZONE_XY)})")
    x, y = PLACE_ZONE_XY[keyword]
    z = PICK_POS[2] if z is None else z
    if ori is None:
        ori = PICK_POS[3:6]
    elif ori == "down":
        ori = GRIPPER_DOWN_ORI_DEG
    return [x, y, z] + list(ori)


def gravity_corrected_tau(n=QUICK_CHECK_SAMPLES):
    '''actual_motor_torque - raw_joint_torque (중력 모델 보정, 진단용).

    raw_joint_torque 는 로봇이 현재 자세에서 계산한 "중력만으로 예상되는 토크"
    모델값이다 (REVIEW.md 1-1). 이걸 actual 에서 빼면 이론상 자세(뻗은 정도)로
    인한 토크 변화가 줄어들어 자세-토크 혼입 문제를 완화할 수 있다.

    ⚠️ 완전한 해결책 아님 — TOOL_COG 가 아직 근사값이고, 이 프로젝트 전체의
    핵심 노이즈원인 마찰 히스테리시스(~3 Nm)는 중력 모델이 애초에 반영하지
    못한다. quick_torque_check() 를 대체하는 게 아니라, 실기에서 두 값을 나란히
    찍어보고 어느 쪽이 자세 변화에 더 안정적인지 비교하는 용도.
    '''
    rows = []
    for _ in range(n):
        d = read_data_rt()
        if d is not None:
            rows.append(d.actual_motor_torque[WEIGHT_JOINT] - d.raw_joint_torque[WEIGHT_JOINT])
        time.sleep(0.04)
    if not rows:
        raise RuntimeError("read_data_rt 가 데이터를 주지 않습니다")
    return statistics.median(rows)


def quick_torque_check(n=QUICK_CHECK_SAMPLES):
    '''dither 없이 J2 토크만 빠르게 본다 (~0.3초).

    마찰 히스테리시스 때문에 절대 무게 추정에는 못 쓴다 (FINDINGS_85mm.md §4).
    값 자체가 아니라 '직전 기준선 대비 크게 벗어났는지'만 본다.
    '''
    return motor_torque(n)[WEIGHT_JOINT]


class DroppedError(RuntimeError):
    '''이동 중 물체를 놓친 것으로 판단됨.'''


class OutOfWorkspaceError(RuntimeError):
    '''목표 좌표가 작업공간을 벗어나거나, 한 번에 너무 멀리 이동하려 함.'''


def check_workspace(pose):
    '''pose[:3] 가 작업공간 안에 있는지, 현재 위치에서 한 번에 너무 멀리 가지
    않는지 확인한다. 범위·최대 이동거리는 OCR팀이 실제로 쓰는 값을 그대로 썼다
    (WORKSPACE_MIN_MM/MAX_MM, MAX_STEP_MM — 우리가 새로 추측한 값이 아님).
    '''
    xyz = list(pose[:3])
    lo, hi = WORKSPACE_MIN_MM, WORKSPACE_MAX_MM
    if not all(lo[i] <= xyz[i] <= hi[i] for i in range(3)):
        raise OutOfWorkspaceError(f"목표 {xyz} 가 작업공간({lo} ~ {hi}) 밖입니다")

    cur, _ = get_current_posx()
    dist = sum((a - b) ** 2 for a, b in zip(xyz, cur[:3])) ** 0.5
    if dist > MAX_STEP_MM:
        raise OutOfWorkspaceError(
            f"한 번에 {dist:.0f}mm 이동은 MAX_STEP_MM({MAX_STEP_MM})을 넘습니다")


def move_to_checked(pose, held_tau, vel=None, checkpoints=TRANSIT_CHECKPOINTS):
    '''목적지까지 나눠서 이동하며, 매 구간 도착마다 놓치지 않았는지 빠르게 확인한다.

    기준선을 원점(held_tau)에 고정하지 않고 매 구간 최신 값으로 갱신한다 —
    자세 변화로 인한 토크 drift는 완만하게 누적되지만 진짜 놓침은 한 구간
    안에서 급변하므로, "직전 구간 대비 급변"만 잡으면 자세 문제로 인한 오탐을
    줄일 수 있다 (완전히 없애진 못 함 — §5 한계 참고). 마지막 구간에서 측정한
    tau 를 반환하므로, 호출부(release_confirmed 등)는 원점 대신 목적지 근처
    기준선을 쓸 수 있다.
    '''
    check_workspace(pose)
    start, _ = get_current_posx()
    start, dest = list(start), list(pose)
    vel = vel or SPEED_L
    baseline = held_tau

    for i in range(1, checkpoints + 1):
        frac = i / checkpoints
        way = [s + frac * (d - s) for s, d in zip(start[:3], dest[:3])] + list(dest[3:])
        movel(posx(way), vel=vel, acc=vel, ref=DR_BASE)
        wait(QUICK_CHECK_SETTLE_S)
        tau = quick_torque_check()
        delta = tau - baseline
        if abs(delta) > DROP_THRESHOLD_NM:
            raise DroppedError(
                f"이동 중 토크 이상 (직전 구간 {baseline:+.3f} → 지금 {tau:+.3f} Nm, "
                f"구간 {i}/{checkpoints})")
        print(f"    체크포인트 {i}/{checkpoints} 통과 — tau {tau:+.3f} Nm (직전 구간 대비 Δ{delta:+.3f})")
        baseline = tau   # 다음 구간은 이번 값 기준으로 비교 (자세 drift 누적 방지)
    wait(0.7)
    return baseline


def release_confirmed(baseline_tau, max_retries=RELEASE_RETRIES):
    '''release() 를 보내고, 토크가 안 바뀌면 명령이 씹혔다고 보고 다시 보낸다.

    그리퍼는 위치 피드백이 없어(GripperCmd.srv 는 제어 전용) '진짜 열렸는지'를
    직접 볼 수 없다. 실기에서 gripper_cmd 를 몇 번 다시 보내야 먹힌 사례가 있어
    (set_robot_mode() 가 비동기라 폴링 확인이 필요했던 것과 같은 계열의 명령
    유실로 추정) 재전송한다.
    '''
    for attempt in range(max_retries + 1):
        release()
        wait(RELEASE_RETRY_WAIT_S)
        tau = quick_torque_check()
        if abs(tau - baseline_tau) >= DROP_THRESHOLD_NM * 0.5:
            return True, attempt
        if attempt < max_retries:
            print(f"  ↻ release 재전송 ({attempt+1}/{max_retries}) — 토크 변화 없음")
    return False, max_retries


In [ ]:
class TransportTest:
    '''grasp_v2.ipynb 의 함수(move_to, grip_at, release 등)는 그대로 두고,
    이동 안전 기능을 move='1'/'2'/'3' 단계로 켜서 검증한다. weight_calib.json
    (=scale.measure()) 이 없어도 고정 current 로 동작한다.

    move
        '1'  근거리 단순 이동만 (offset_mm, 체크포인트 없음)
        '2'  move_to_checked() 로 체크포인트 나눠 이동 (놓침 감지 켜짐)
        '3'  place_pos(OCR 좌표) 로 실이동 (체크포인트 켜짐, place_pos 필수)

    DroppedError 가 나면: quick_torque_check() 를 한 번 더 재서 grip 전 기준선
    (empty_tau)과 비교합니다. 거의 같으면 "완전히 놓친 것"으로 보고 PICK_POS 로
    복귀(새 좌표 아님, 이미 검증된 위치), 다르면 "뭔가 불안정하게 물려있는 것"으로
    보고 그 자리에서 멈춥니다.
    vel
        None       기본. SPEED_L 고정으로만 이동
        'measure'  위와 동일하게 움직이되 결과를 transport_test_log.json 에 기록
                   (자동 판정은 안 함 — 나중에 mark_slip() 으로 직접 라벨링)
    retry
        'grip'(기본)  리프트 직후 quick_torque_check 가 grip 전 기준선과 거의 같으면
                      (=아무것도 안 잡힌 것으로 보이면) CURRENT_STEP 만큼 올려 재시도
        None          재시도 없이 한 번만
    return_home
        True(기본)  release + 복귀 후 PICK_POS 로 다시 이동 (새 좌표 아님, 이미 매번
                    쓰는 검증된 위치라 안전). 다음 사이클 대비 + 카메라 시야 확보용
        False       현재 위치(place 위쪽)에서 그대로 종료
    '''

    STAGES = {"1", "2", "3"}

    def __init__(self, move="1", vel=None, retry="grip", width_mm=85.0,
                 offset_mm=None, place_pos=None, current=None,
                 checkpoints=TRANSIT_CHECKPOINTS, note="", return_home=True):
        if move not in self.STAGES:
            raise ValueError(f"move='{move}' 알 수 없음 (1/2/3)")
        if isinstance(place_pos, str):
            place_pos = resolve_place_pos(place_pos)   # '수원'/'서울' 같은 키워드 지원
        if move == "3" and place_pos is None:
            raise ValueError("move='3' 은 place_pos(OCR 좌표 또는 '수원'/'서울') 가 필요합니다")

        self.move = move
        self.vel_mode = vel
        self.retry_mode = retry
        self.width_mm = width_mm
        self.offset_mm = offset_mm
        self.place_pos = place_pos
        self.current = current or TEST_CURRENT
        self.checkpoints = checkpoints
        self.note = note
        self.return_home = return_home
        self.result = self.run()

    def _grip_and_lift(self):
        stroke = stroke_from_width_mm(self.width_mm - SQUEEZE_MM)
        cur = self.current

        release()
        move_to(PICK_POS)
        self.empty_tau = quick_torque_check()  # PICK_POS 에서 잰 빈 그리퍼 기준선 (retry/드롭 분기 판정용)
        empty_tau = self.empty_tau

        attempts = 0
        while True:
            attempts += 1
            grip_at(stroke, cur, context=f"test:move={self.move}")
            time.sleep(1)

            lifted = list(PICK_POS)
            lifted[2] += LIFT_MM
            move_to(lifted)
            held_tau = quick_torque_check()

            looks_empty = abs(held_tau - empty_tau) < DROP_THRESHOLD_NM * 0.5
            if not looks_empty or self.retry_mode != "grip" or cur >= CURRENT_MAX:
                break
            cur = min(cur + CURRENT_STEP, CURRENT_MAX)
            print(f"  ↻ 리프트 재시도 {attempts}: current {cur} (아무것도 안 잡힌 것으로 보임)")
            release()
            move_to(PICK_POS)

        return lifted, held_tau, attempts, (not looks_empty)

    def run(self):
        lifted, held_tau, attempts, held = self._grip_and_lift()
        state = "들림" if held else "못 들었을 수 있음"
        print(f"[move={self.move}] 기준선 tau {held_tau:+.3f} ({state}, {attempts}회 시도)")

        vel = SPEED_L
        dropped, drop_reason = False, None

        if self.move == "1":
            dest = list(lifted)
            dest[0] += self.offset_mm or SHORT_MOVE_OFFSET_MM
            move_to(dest, vel=vel)
        else:
            dest = self.place_pos
            if dest is None:
                dest = list(lifted)
                dest[0] += self.offset_mm or WAYPOINT_OFFSET_MM
            try:
                move_to_checked(dest, held_tau=held_tau, vel=vel, checkpoints=self.checkpoints)
            except DroppedError as e:
                dropped, drop_reason = True, str(e)
                print("  ❌ 이동 중 이상 감지:", e)
                after_tau = quick_torque_check()
                if abs(after_tau - self.empty_tau) < DROP_THRESHOLD_NM * 0.5:
                    print("     완전히 놓친 것으로 보입니다 (그리퍼가 비어있음) — PICK_POS 로 복귀합니다.")
                    move_to(PICK_POS)
                else:
                    print("     아직 뭔가 물려있는 것으로 보입니다 — 더 위험해질 수 있어 지금 위치에서 멈춥니다.")

        released, release_retries = False, 0
        if not dropped:
            released, release_retries = release_confirmed(held_tau)
            if not released:
                print(f"  ⚠️ release {RELEASE_RETRIES}회 재전송에도 토크가 그대로입니다.")
            up = list(dest)
            up[2] += LIFT_MM
            move_to(up)

            if self.return_home:
                move_to(PICK_POS)   # 다음 사이클 대비 + 카메라 시야 확보. PICK_POS 는 이미 검증된 위치라 새 좌표 위험 없음

        result = {
            "move": self.move, "vel": vel, "width_mm": self.width_mm,
            "current": self.current, "grip_attempts": attempts, "held": held,
            "dropped_in_transit": dropped, "drop_reason": drop_reason,
            "released": released, "release_retries": release_retries,
            "note": self.note, "t": datetime.now().isoformat(timespec="seconds"),
        }

        if self.vel_mode == "measure":
            self._log(result)

        return result

    @staticmethod
    def _log(entry):
        entries = json.loads(TRANSPORT_LOG_PATH.read_text()) if TRANSPORT_LOG_PATH.exists() else []
        entry = dict(entry)
        entry["index"] = len(entries)
        entry["slipped"] = None    # 나중에 mark_slip() 으로 채움
        entries.append(entry)
        TRANSPORT_LOG_PATH.write_text(json.dumps(entries, indent=2, ensure_ascii=False))
        print(f"  → transport_test_log.json 에 기록 (#{entry['index']})")


def review_transport_log():
    if not TRANSPORT_LOG_PATH.exists():
        print("기록 없음")
        return
    entries = json.loads(TRANSPORT_LOG_PATH.read_text())
    print(f"{'idx':>3} | {'move':>4} | {'vel':>7} | {'폭':>5} | {'놓침':>6} | {'미끄러짐(라벨)':>14} | note")
    for e in entries:
        slip = {True: "예", False: "아니오", None: "-"}[e.get("slipped")]
        print(f"{e['index']:>3} | {e['move']:>4} | {e['vel']:>7} | {e['width_mm']:>5.0f} | "
              f"{'예' if e['dropped_in_transit'] else '아니오':>6} | {slip:>14} | {e['note']}")


def mark_slip(i, slipped, note=""):
    '''transport_test_log.json 의 i번 기록에 실제로 미끄러졌는지 직접 라벨링한다.'''
    entries = json.loads(TRANSPORT_LOG_PATH.read_text())
    entries[i]["slipped"] = bool(slipped)
    if note:
        entries[i]["note"] = note
    TRANSPORT_LOG_PATH.write_text(json.dumps(entries, indent=2, ensure_ascii=False))
    print(f"[{i}] slipped={slipped} 로 기록")


### 사용 예

```python
# 1순위: 근거리 단순 이동
TransportTest(move='1', width_mm=85)

# 체크포인트 이동 확인 (거리 직접 지정 가능)
TransportTest(move='2', width_mm=85, offset_mm=400)

# OCR 좌표 들어오면 (직접 좌표)
TransportTest(move='3', width_mm=85, place_pos=[x, y, z, rx, ry, rz])

# OCR 놓기 구역 키워드로도 가능 (X/Y만 OCR팀 값, Z·자세는 우리 PICK_POS 기준)
# ⚠️ '경기'는 아직 OCR팀 코드에 없음. 수원 좌표는 팀 확인 전까지 신뢰하지 말 것
TransportTest(move='3', width_mm=85, place_pos='서울')

# 속도 데이터 쌓기 (여러 물체·거리로 반복)
TransportTest(move='2', width_mm=75, vel='measure', note='75mm 상자')
review_transport_log()
mark_slip(0, slipped=False)   # 육안으로 안 미끄러진 거 확인했으면
```